In [41]:
import requests
import pandas as pd
import time
import ast
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [28]:
url = "https://api.opendota.com/api/publicMatches"
all_matches = []
params = {}
print("Pinging the OpenDota API...")

Pinging the OpenDota API...


In [29]:
response = requests.get(url)
for i in range(50):
    response = requests.get(url, params = params)
    
    if response.status_code == 200:

        batch_data = response.json()
        all_matches.extend(batch_data)
        
        oldest_match_id = min([match['match_id'] for match in batch_data])
        
        params = {'less_than_match_id': oldest_match_id}
        
        print(f"Batch {i+1}/50 secured. Sleeping for 2 seconds...")
        
        time.sleep(2)

    else:
        print(f"Uh oh, the server got mad on batch {i+1}. Status code: {response.status_code}")
        break

Batch 1/50 secured. Sleeping for 2 seconds...
Batch 2/50 secured. Sleeping for 2 seconds...
Batch 3/50 secured. Sleeping for 2 seconds...
Batch 4/50 secured. Sleeping for 2 seconds...
Batch 5/50 secured. Sleeping for 2 seconds...
Batch 6/50 secured. Sleeping for 2 seconds...
Batch 7/50 secured. Sleeping for 2 seconds...
Batch 8/50 secured. Sleeping for 2 seconds...
Batch 9/50 secured. Sleeping for 2 seconds...
Batch 10/50 secured. Sleeping for 2 seconds...
Batch 11/50 secured. Sleeping for 2 seconds...
Batch 12/50 secured. Sleeping for 2 seconds...
Batch 13/50 secured. Sleeping for 2 seconds...
Batch 14/50 secured. Sleeping for 2 seconds...
Batch 15/50 secured. Sleeping for 2 seconds...
Batch 16/50 secured. Sleeping for 2 seconds...
Batch 17/50 secured. Sleeping for 2 seconds...
Batch 18/50 secured. Sleeping for 2 seconds...
Batch 19/50 secured. Sleeping for 2 seconds...
Batch 20/50 secured. Sleeping for 2 seconds...
Batch 21/50 secured. Sleeping for 2 seconds...
Batch 22/50 secured. S

In [ ]:
df = pd.DataFrame(all_matches)

df = df.drop_duplicates(subset = 'match_id')

print(f"\nHarvest complete! We collected {len(df)} unique matches.")

df.to_csv("dota_5k_matches.csv", index=False)
print("Data safely locked away in dota_5k_matches.csv")


Harvest complete! We collected 5000 unique matches.
Data safely locked away in dota_5k_matches.csv


In [33]:
print("Loading the massive dataset...")
df_5k = pd.read_csv("dota_5k_matches.csv")

Loading the massive dataset...


In [34]:
df_5k['radiant_team'] =  df_5k['radiant_team'].apply(ast.literal_eval)
df_5k['dire_team'] = df_5k['dire_team'].apply(ast.literal_eval)

print("Data loaded successfully! Here is what the target variable and features look like:")
display(df_5k[['match_id', 'radiant_win', 'radiant_team', 'dire_team']].head())

Data loaded successfully! Here is what the target variable and features look like:


,match_id,radiant_win,radiant_team,dire_team
0,8715095581,False,"[102, 79, 2, 42, 82]","[8, 105, 29, 67, 23]"
1,8715094411,True,"[84, 92, 102, 99, 71]","[8, 39, 120, 87, 5]"
2,8715093917,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
3,8715093899,True,"[87, 12, 14, 36, 40]","[6, 51, 108, 52, 64]"
4,8715092921,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"


In [35]:
print("Building the hero matrix... this might take a few seconds.")

matrix_rows = []

for i, row in df_5k.iterrows():
    
    # Create empty dictionary for match
    match_features  =  {}
    
    # +1 for Radiant hero
    for hero in row['radiant_team']:
        match_features[f'hero_{hero}'] = 1
    
    # -1 for Dire hero
    for hero in row['dire_team']:
        match_features[f'hero_{hero}'] = -1
        
    matrix_rows.append(match_features)
    
hero_df = pd.DataFrame(matrix_rows).fillna(0)

final_df = pd.concat([df_5k[['match_id', 'radiant_win']], hero_df], axis=1)

print("Matrix complete! Here is the shape of our new dataset (Rows, Columns):")
print(final_df.shape)
display(final_df.head())

Building the hero matrix... this might take a few seconds.
Matrix complete! Here is the shape of our new dataset (Rows, Columns):
(5000, 130)


,match_id,radiant_win,hero_102,hero_79,hero_2,hero_42,hero_82,hero_8,hero_105,hero_29,...,hero_112,hero_103,hero_57,hero_3,hero_78,hero_90,hero_73,hero_46,hero_43,hero_69
0,8715095581,False,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,8715094411,True,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,8715093917,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8715093899,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8715092921,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
X = final_df.drop(columns=['match_id','radiant_win'])
y = final_df['radiant_win']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 12)

print("Data successfully split!")
print(f"Training matches: {X_train.shape[0]}")
print(f"Testing matches: {X_test.shape[0]}")

Data successfully split!
Training matches: 4000
Testing matches: 1000


In [38]:
model = LogisticRegression()

model.fit(X_train, y_train)

preds = model.predict(X_test)

score = accuracy_score(y_test, preds)
print(f"Model Accuracy: {score * 100:.2f}%")

Model Accuracy: 55.20%


In [39]:
coefficients = pd.DataFrame({
    'Hero': X_train.columns,
    'Weight': model.coef_[0] 
})

print("Top 5 Heroes the model thinks are strong(Highest Positive Weights):")
display(coefficients.sort_values(by='Weight', ascending=False).head(5))

print("\nTop 5 Heroes the model thinks are weak (Lowest Negative Weights):")
display(coefficients.sort_values(by='Weight', ascending=True).head(5))

Top 5 Heroes the model thinks are strong(Highest Positive Weights):


,Hero,Weight
18,hero_0,1.651790
11,hero_92,0.484208
122,hero_78,0.374883
37,hero_81,0.318391
125,hero_46,0.308487



Top 5 Heroes the model thinks are weak (Lowest Negative Weights):


,Hero,Weight
90,hero_114,-0.525569
15,hero_120,-0.494275
40,hero_72,-0.350412
86,hero_131,-0.343334
56,hero_53,-0.343208


In [44]:
print('Planting the Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=12,
    n_jobs=-1
)

print("Training the trees ...")
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_score = accuracy_score(y_test, rf_predictions)

print("\n--- FINAL RESULTS ---")
print(f"New Random Forest Accuracy: {rf_score * 100:.2f}%")

importances = pd.DataFrame({
    'Hero': X_train.columns,
    'Importance': rf_model.feature_importances_
})

print("\nTop 5 Most Decisive Heroes (The ones swinging the win probability the most):")
display(importances.sort_values(by='Importance', ascending=False).head(5))

Planting the Random Forest...
Training the trees ...

--- FINAL RESULTS ---
New Random Forest Accuracy: 54.30%

Top 5 Most Decisive Heroes (The ones swinging the win probability the most):


,Hero,Importance
20,hero_14,0.016774
34,hero_26,0.015734
32,hero_35,0.015323
10,hero_84,0.014669
60,hero_86,0.014177
